<a href="https://colab.research.google.com/github/purveshvasantwagh-alt/digital-pathology-cell-analysis/blob/cell-analysis-pipeline/Cell_Segmentation_and_Classification_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision opencv-python-headless matplotlib scikit-learn grad-cam kagglehub

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models, transforms
import kagglehub

print("Setup completed. GPU active:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Setup completed. GPU active: True


In [2]:
# Downloads Blood Cell Images dataset automatically
dataset_path = kagglehub.dataset_download("paultimothymooney/blood-cells")
print("Dataset location:", dataset_path)

100%|██████████| 108M/108M [00:05<00:00, 21.5MB/s]

Extracting files...


Dataset location: /root/.cache/kagglehub/datasets/paultimothymooney/blood-cells/versions/6


In [3]:
from pathlib import Path

root = Path(dataset_path)
print("Dataset folder:", root)
for path in list(root.rglob("*"))[:80]:
    print(path.relative_to(root), "DIR" if path.is_dir() else "FILE")

Dataset folder: /root/.cache/kagglehub/datasets/paultimothymooney/blood-cells/versions/6
dataset-master DIR
dataset2-master DIR
dataset-master/dataset-master DIR
dataset2-master/dataset2-master DIR
dataset2-master/dataset2-master/images DIR
dataset2-master/dataset2-master/labels.csv FILE
dataset2-master/dataset2-master/images/TEST_SIMPLE DIR
dataset2-master/dataset2-master/images/TEST DIR
dataset2-master/dataset2-master/images/TRAIN DIR
dataset2-master/dataset2-master/images/TEST_SIMPLE/MONOCYTE DIR
dataset2-master/dataset2-master/images/TEST_SIMPLE/EOSINOPHIL DIR
dataset2-master/dataset2-master/images/TEST_SIMPLE/LYMPHOCYTE DIR
dataset2-master/dataset2-master/images/TEST_SIMPLE/NEUTROPHIL DIR
dataset2-master/dataset2-master/images/TEST/MONOCYTE DIR
dataset2-master/dataset2-master/images/TEST/EOSINOPHIL DIR
dataset2-master/dataset2-master/images/TEST/LYMPHOCYTE DIR
dataset2-master/dataset2-master/images/TEST/NEUTROPHIL DIR
dataset2-master/dataset2-master/images/TRAIN/MONOCYTE DIR
datas

In [4]:
from pathlib import Path
import cv2

base = Path(dataset_path)
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
splits = ["TRAIN", "TEST", "TEST_SIMPLE"]

for images_dir in sorted(base.rglob("images")):
    if not any((images_dir / split).is_dir() for split in splits):
        continue

    print(f"\nImages folder: {images_dir.relative_to(base)}")

    for split in splits:
        split_dir = images_dir / split
        if not split_dir.is_dir():
            continue

        print(f"  {split}:")
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            files = [
                p for p in class_dir.iterdir()
                if p.is_file() and p.suffix.lower() in image_extensions
            ]
            sample = cv2.imread(str(files[0])) if files else None
            dimensions = sample.shape[:2] if sample is not None else "no readable sample"
            print(f"    {class_dir.name}: {len(files)} images; sample height,width={dimensions}")

for csv_path in sorted(base.rglob("labels.csv")):
    print(f"\nCSV found: {csv_path.relative_to(base)}")
    with open(csv_path, encoding="utf-8") as f:
        for _, line in zip(range(6), f):
            print(" ", line.rstrip())


Images folder: dataset2-master/dataset2-master/images
  TRAIN:
    EOSINOPHIL: 2497 images; sample height,width=(240, 320)
    LYMPHOCYTE: 2483 images; sample height,width=(240, 320)
    MONOCYTE: 2478 images; sample height,width=(240, 320)
    NEUTROPHIL: 2499 images; sample height,width=(240, 320)
  TEST:
    EOSINOPHIL: 623 images; sample height,width=(240, 320)
    LYMPHOCYTE: 620 images; sample height,width=(240, 320)
    MONOCYTE: 620 images; sample height,width=(240, 320)
    NEUTROPHIL: 624 images; sample height,width=(240, 320)
  TEST_SIMPLE:
    EOSINOPHIL: 13 images; sample height,width=(240, 320)
    LYMPHOCYTE: 6 images; sample height,width=(240, 320)
    MONOCYTE: 4 images; sample height,width=(240, 320)
    NEUTROPHIL: 48 images; sample height,width=(240, 320)

CSV found: dataset-master/dataset-master/labels.csv
  ,Image,Category
  ,0,NEUTROPHIL
  ,1,NEUTROPHIL
  ,2,NEUTROPHIL
  ,3,NEUTROPHIL
  ,4,NEUTROPHIL

CSV found: dataset2-master/dataset2-master/labels.csv
  ,Imag

In [5]:
%cd /content
!git clone --single-branch --branch cell-analysis-pipeline https://github.com/purveshvasantwagh-alt/digital-pathology-cell-analysis.git
%cd /content/digital-pathology-cell-analysis
!python -m cell_analysis.train --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --epochs 5 --batch-size 32

/content
Cloning into 'digital-pathology-cell-analysis'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 3), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 11.43 KiB | 2.86 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/digital-pathology-cell-analysis
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 174MB/s]
Device: cuda
Classes: ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']
Epoch 1/5 | train loss=1.0771, train accuracy=0.5418 | validation loss=0.8195, validation accuracy=0.7082
Saved best checkpoint to: models/resnet18_best.pt
Epoch 2/5 | train loss=0.8298, train accuracy=0.6762 | validation loss=0.6931, validation accuracy=0.7497
Saved best checkpoint to: models/resnet18_best.pt
Epoch 3/5 | train loss=0.7452, tra

In [6]:
%cd /content/digital-pathology-cell-analysis
!git pull origin cell-analysis-pipeline
!python -m cell_analysis.evaluate --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --checkpoint models/resnet18_best.pt --report-dir reports

/content/digital-pathology-cell-analysis
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.33 KiB | 1.33 MiB/s, done.
From https://github.com/purveshvasantwagh-alt/digital-pathology-cell-analysis
 * branch            cell-analysis-pipeline -> FETCH_HEAD
   2731f42..2a13c60  cell-analysis-pipeline -> origin/cell-analysis-pipeline
Updating 2731f42..2a13c60
Fast-forward
 cell_analysis/evaluate.py | 96 +++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 96 insertions(+)
 create mode 100644 cell_analysis/evaluate.py
Test images evaluated: 2487
              precision    recall  f1-score   support

  EOSINOPHIL     0.6584    0.5136    0.5771       623
  LYMPHOCYTE     0.7734    0.7984    0.7857       620
    MONOCYTE     0.6548    0.6210    0.6374       620
  NEUTROPHIL     0.5990    0.7420    0.6628

In [7]:
%cd /content/digital-pathology-cell-analysis
!git pull origin cell-analysis-pipeline
!python -m cell_analysis.evaluate --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --checkpoint models/resnet18_best.pt --report-dir reports --split validation

/content/digital-pathology-cell-analysis
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.02 KiB | 1.02 MiB/s, done.
From https://github.com/purveshvasantwagh-alt/digital-pathology-cell-analysis
 * branch            cell-analysis-pipeline -> FETCH_HEAD
   2a13c60..29bf953  cell-analysis-pipeline -> origin/cell-analysis-pipeline
Updating 2a13c60..29bf953
Fast-forward
 cell_analysis/evaluate.py | 82 ++++++++++++++++++++++++++++++++++++-----------
 1 file changed, 63 insertions(+), 19 deletions(-)
Device: cuda
Checkpoint: models/resnet18_best.pt
Validation images evaluated: 1494
              precision    recall  f1-score   support

  EOSINOPHIL     0.8244    0.6133    0.7034       375
  LYMPHOCYTE     0.8131    0.9005    0.8546       372
    MONOCYTE     0.7926    0.8629    0.8263       372
  NEUTROPHIL     

In [9]:
%cd /content/digital-pathology-cell-analysis
!git pull origin cell-analysis-pipeline
!python -m cell_analysis.train --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --epochs 8 --batch-size 32 --checkpoint models/resnet18_layer4_best.pt --fine-tune-layer4

/content/digital-pathology-cell-analysis
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.13 KiB | 2.13 MiB/s, done.
From https://github.com/purveshvasantwagh-alt/digital-pathology-cell-analysis
 * branch            cell-analysis-pipeline -> FETCH_HEAD
   29bf953..8b3cf7a  cell-analysis-pipeline -> origin/cell-analysis-pipeline
Updating 29bf953..8b3cf7a
Fast-forward
 cell_analysis/train.py | 157 ++++++++++++++++++++++++++++++++++++++++---------
 1 file changed, 128 insertions(+), 29 deletions(-)
Device: cuda
Classes: ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']
Fine-tuning layer4: True
Training images: 8463
Validation images: 1494
Epoch 1/8 | train loss=0.3350, train accuracy=0.8687 | validation loss=0.0632, validation accuracy=0.9772
Saved best checkpoint to: models/resnet18_layer4_best.pt
Epoch

In [10]:
%cd /content/digital-pathology-cell-analysis
!python -m cell_analysis.evaluate --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --checkpoint models/resnet18_layer4_best.pt --report-dir reports/layer4-experiment --split validation
!python -m cell_analysis.evaluate --images-root "{dataset_path}/dataset2-master/dataset2-master/images" --checkpoint models/resnet18_layer4_best.pt --report-dir reports/layer4-experiment --split test

/content/digital-pathology-cell-analysis
Device: cuda
Checkpoint: models/resnet18_layer4_best.pt
Validation images evaluated: 1494
              precision    recall  f1-score   support

  EOSINOPHIL     0.9921    1.0000    0.9960       375
  LYMPHOCYTE     1.0000    1.0000    1.0000       372
    MONOCYTE     1.0000    1.0000    1.0000       372
  NEUTROPHIL     1.0000    0.9920    0.9960       375

    accuracy                         0.9980      1494
   macro avg     0.9980    0.9980    0.9980      1494
weighted avg     0.9980    0.9980    0.9980      1494

Confusion matrix (rows = actual, columns = predicted):
[[375   0   0   0]
 [  0 372   0   0]
 [  0   0 372   0]
 [  3   0   0 372]]
Reports saved in: reports/layer4-experiment/validation
Device: cuda
Checkpoint: models/resnet18_layer4_best.pt
Test images evaluated: 2487
              precision    recall  f1-score   support

  EOSINOPHIL     0.7593    0.7191    0.7387       623
  LYMPHOCYTE     0.9527    0.9097    0.9307       620


In [11]:
from collections import defaultdict
from pathlib import Path
import hashlib

from sklearn.model_selection import train_test_split
from torchvision.datasets import ImageFolder

images_root = (
    Path(dataset_path)
    / "dataset2-master"
    / "dataset2-master"
    / "images"
)

train_dataset = ImageFolder(images_root / "TRAIN")
indices = list(range(len(train_dataset)))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.15,
    random_state=42,
    stratify=train_dataset.targets,
)

test_dataset = ImageFolder(images_root / "TEST")

split_paths = {
    "train": [Path(train_dataset.samples[i][0]) for i in train_indices],
    "validation": [Path(train_dataset.samples[i][0]) for i in val_indices],
    "test": [Path(path) for path, _ in test_dataset.samples],
}

hash_to_splits = defaultdict(set)
hash_to_paths = defaultdict(list)

for split_name, paths in split_paths.items():
    for path in paths:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        hash_to_splits[digest].add(split_name)
        hash_to_paths[digest].append(str(path))

duplicates = {
    digest: hash_to_paths[digest]
    for digest, splits in hash_to_splits.items()
    if len(splits) > 1
}

print("Exact duplicate images crossing splits:", len(duplicates))
for paths in list(duplicates.values())[:10]:
    print(paths)

Exact duplicate images crossing splits: 0


**Exploratory watershed segmentation prototype:** Applies thresholding, morphology, and watershed to an image. It has not been validated against annotated masks; its cell counts are not validated measurements. It is not connected to the classifier.**bold text**

In [ ]:
def process_cell_segmentation(image_path):
    # Read image and convert color spaces
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Noise reduction and adaptive thresholding
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological operations to clean background noise
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

    # Distance transform & Watershed segmentation for overlapping cell boundaries
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)

    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    markers = cv2.watershed(img_rgb, markers)
    cell_count = len(np.unique(markers)) - 2

    return img_rgb, markers, cell_count